# Ingest Drivers files (multi json)

In [0]:
dbutils.widgets.text('p_batch_id', '')

In [0]:
v_batch_id = dbutils.widgets.get('p_batch_id')
v_batch_id

In [0]:
%run ../00-common/01.environment-config


In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_path = f'{landing_folder_path}/{v_batch_id}/drivers.json'
table_name = f'{catalog_name}.{bronze_schema}.drivers'

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

drivers_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType())
])

In [0]:
drivers_df = (
    spark.read
        .format('json')
        .schema(drivers_schema)
        .load(source_path)
)

In [0]:
display(drivers_df)

In [0]:
drivers_df_final = add_ingestion_metadata(drivers_df)

In [0]:
write_to_bronze (
    final_df = drivers_df_final,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
# (
#     drivers_df_final.write
#         .format('delta')
#         .mode('overwrite')
#         .saveAsTable(table_name)
# )

In [0]:
%sql
SELECT * FROM formula1.bronze.drivers